In [9]:
from semanticscholar import SemanticScholar
from habanero import Crossref
from itertools import product
import pandas as pd
import json


In [16]:
primary_keywords = " | ".join(
    ["Generative AI", "large language model", "codex", "gpt-3", "gpt-4"]
)  # and so on...
secondary_keywords = " | ".join(
    [
        "overreliance",
        "misinformation",
        "accessbility",
        "privacy",
        "enviromental",
        "explainability",
        "trustworthy",
        "responsible",
    ]
)  # and so on...

year = "2019-"

all_keywords = f"({primary_keywords}) + ({secondary_keywords})"

In [23]:
# Prepare DataFrame to store the results
columns = [
    'PaperTitle',
    'DOI',
    'Authors',
    'Abstract',
    'Publisher',
    'SemanticScholarUrl',
    'DoiUrl',
    'PublicationDate',
    'FieldOfStudy',
    'Conference-Journal',
    'PublicationTypes',
    'SearchString',
    'CitationCount',
    'SearchedFrom'
]

In [38]:
import requests
from requests import HTTPError

results = []
crossref = Crossref()
query = all_keywords
fields = [
    "title",
    "externalIds",
    "authors",
    "abstract",
    "url",
    "publicationDate",
    "fieldsOfStudy",
    "venue",
    "publicationTypes",
    "citationCount",
]

# send http request to api
api_url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
headers = {"Content-Type": "application/json"}
params = {"query": query, "year": year, "fields": ",".join(fields)}
response = requests.get(api_url, headers=headers, params=params)

# parse response
limit = 20
counter = 0
response_json = response.json()
sch_results = response_json["data"]

for sch_paper in sch_results:
    if counter >= limit:
        break
    try:
        crossref_paper = crossref.works(ids=sch_paper["externalIds"].get("DOI"))
    except HTTPError as e:
        print(
            f"An HTTP error occurred for query '{query}': {e}, error code: {e.response.status_code}"
        )
        crossref_paper = None
    except Exception as e:
        print(f"An error occurred for query '{query}': {e}")
        crossref_paper = None

    title = sch_paper["title"]
    doi = sch_paper["externalIds"].get("DOI")

    authors = None
    # author name and affiliation
    if crossref_paper is not None:
        authors = crossref_paper.get("message").get("author")
    if authors is not None:
        for i in range(len(authors)):
            author = authors[i]
            author_name = author.get("given", "") + " " + author.get("family", "")
            affiliation = author.get("affiliation", "No Affiliation")
            affiliations = author.get("affiliation", [])
            school_names = (
                [affil.get("name") for affil in affiliations]
                if affiliations
                else ["No Affiliation"]
            )
            # Create a new dictionary with only 'name' and 'affiliation'
            authors[i] = {
                "name": author_name.strip(),
                "affiliation": school_names,
            }
    else:
        authors = sch_paper["authors"]
        for i in range(len(sch_paper["authors"])):
            author = sch_paper["authors"][i]
            sch_paper["authors"][i] = {
                "name": author.get("name", "No Name"),
                "affiliation": author.get("affiliation", "No Affiliation"),
            }

    abstract = sch_paper["abstract"]
    sch_url = sch_paper["url"]
    doi_url = f"https://doi.org/{doi}"
    publication_date = sch_paper["publicationDate"]
    fields_of_study = sch_paper["fieldsOfStudy"]
    venue = sch_paper["venue"]

    # publisher
    if crossref_paper is not None:
        publisher = crossref_paper.get("message").get("publisher")
    elif "arxiv" in doi.lower():
        publisher = "arXiv"
    else:
        publisher = None

    # paper type
    if crossref_paper is not None:
        paper_type = [crossref_paper.get("message").get("type")]
    else:
        paper_type = sch_paper["publicationTypes"]

    citation_count = sch_paper["citationCount"]
    # TODO: keywords missing
    # TODO: paper type is conference/journal for arxiv papers
    # TODO: conference-journal name mismatch with publisher, i.e., for paper with name"ChatGPT in education: A discourse analysis of worries and concerns on social media", the conference name is "International Conference on Artificial Intelligence in Education", but the publisher is "Arxiv" (becauseit queryed from arxiv), need "Springer" instead.

    new_paper = {
        "PaperTitle": title,
        "DOI": doi,
        "Authors": authors,
        "Abstract": abstract,
        "Publisher": publisher,
        "SemanticScholarUrl": sch_url,
        "DoiUrl": doi_url,
        "PublicationDate": publication_date,
        "FieldOfStudy": fields_of_study,
        "Conference-Journal": venue,
        "PublicationTypes": paper_type,
        "SearchString": query,
        "CitationCount": citation_count,
        "SearchedFrom": "Semantic Scholar",
    }
    results.append(new_paper)
    counter += 1

results_df = pd.DataFrame(results, columns=columns)
results_df.to_csv("data/initial-scrape-result.csv", index=False)

An HTTP error occurred for query '(Generative AI | large language model | codex | gpt-3 | gpt-4) + (overreliance | misinformation | accessbility | privacy | enviromental | explainability | trustworthy | responsible)': 404 Client Error: Not Found for url: https://api.crossref.org/works/10.48550/arXiv.2309.17157, error code: 404
An HTTP error occurred for query '(Generative AI | large language model | codex | gpt-3 | gpt-4) + (overreliance | misinformation | accessbility | privacy | enviromental | explainability | trustworthy | responsible)': 404 Client Error: Not Found for url: https://api.crossref.org/works/10.48550/arXiv.2211.07517, error code: 404
An HTTP error occurred for query '(Generative AI | large language model | codex | gpt-3 | gpt-4) + (overreliance | misinformation | accessbility | privacy | enviromental | explainability | trustworthy | responsible)': 404 Client Error: Not Found for url: https://api.crossref.org/works/10.48550/arXiv.2306.10062, error code: 404
An HTTP error

In [13]:
# semantic scholar test
sch = SemanticScholar()
sch_results = sch.get_paper("10.1177/030631284014003004")


In [12]:
#corssref test

cr = Crossref()
cr_results = cr.works(query = "Education and Information Technologies : Official Journal of the IFIP technical committee on Education")